# 🎭 Face Swap Live — Google Colab (GPU)

**Passo 0 (obrigatório):** `Runtime → Change runtime type → T4 GPU`

Execute as células em ordem.

In [ ]:
# Célula 1 — Verifica GPU
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    print(r.stdout)
else:
    print('GPU nao encontrada! Va em Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# Célula 2 — Instala dependências
!pip install -q flask flask-socketio eventlet \
    insightface onnxruntime-gpu opencv-python-headless \
    huggingface_hub pillow numpy

# Instala cloudflared (tunnel sem precisar de conta)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('Instalacao concluida!')

In [ ]:
# Célula 3 — Clona / atualiza o repositório
import os
if not os.path.exists('faceSwapLive2'):
    !git clone https://github.com/moniquepavan/faceSwapLive2.git
else:
    !cd faceSwapLive2 && git pull
os.chdir('faceSwapLive2')
print('Diretorio:', os.getcwd())

In [ ]:
# Célula 4 — Baixa o modelo inswapper (~280 MB)
import os, shutil
from huggingface_hub import hf_hub_download

DEST = 'models/inswapper_128.onnx'
os.makedirs('models', exist_ok=True)

if os.path.exists(DEST):
    print(f'Modelo ja existe ({os.path.getsize(DEST)/1e6:.0f} MB) — pulando.')
else:
    token = input('Cole seu token do Hugging Face (huggingface.co/settings/tokens): ').strip()
    print('Baixando...')
    path = hf_hub_download(
        repo_id='hacksider/deep-live-cam',
        filename='inswapper_128_fp16.onnx',
        token=token,
        local_dir='models'
    )
    if os.path.abspath(path) != os.path.abspath(DEST):
        shutil.move(path, DEST)
    print(f'Download concluido! {os.path.getsize(DEST)/1e6:.0f} MB')

In [ ]:
# Célula 5 — Inicia servidor + tunnel público (sem precisar de conta)
import subprocess, threading, time, sys, re

# Inicia Flask em processo separado
server = subprocess.Popen(
    [sys.executable, 'app_colab.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

def print_logs():
    for line in server.stdout:
        print('[srv]', line.rstrip())
threading.Thread(target=print_logs, daemon=True).start()

print('Aguardando Flask iniciar (5s)...')
time.sleep(5)

# Inicia cloudflared tunnel (sem conta, sem token)
cf = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Espera a URL aparecer nos logs
url = None
print('Aguardando URL do tunnel...')
for line in cf.stdout:
    print('[cf]', line.rstrip())
    match = re.search(r'https://[\w\-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        break

print()
print('=' * 60)
print(f'  URL PUBLICA: {url}')
print('=' * 60)
print()
print('1. Abra a URL acima no navegador')
print('2. Aguarde status "Pronto! [CUDA GPU]" (pode levar ~60s)')
print('3. Faca upload de uma foto e inicie a camera')
print()
print('Mantenha esta celula rodando enquanto usar o app.')

# Mantém vivo
try:
    cf.wait()
except KeyboardInterrupt:
    cf.terminate()
    server.terminate()
    print('Encerrado.')